In [1]:
import trimesh
import numpy as np
import cv2
import json
import os
import matplotlib.pyplot as plt

MODEL_PATH = "../data/cad/e-motor.stp"

# Step 1: wide-field detection — angled view, low res
STEP1_DIR        = "../data/templates_wide"
STEP1_ELEVATIONS = [20, 30, 40]
STEP1_AZ_STEP    = 10
STEP1_IMG_SIZE   = 128

# Step 2: close-up precision — top-down, higher res
STEP2_DIR        = "../data/templates_topdown"
STEP2_ELEVATIONS = [0, 5]   # nearly top-down
STEP2_AZ_STEP    = 5
STEP2_IMG_SIZE   = 256

PADDING = 12

# Load mesh
scene = trimesh.load(MODEL_PATH)
mesh  = trimesh.util.concatenate(list(scene.geometry.values())) if isinstance(scene, trimesh.Scene) else scene
mesh.apply_translation(-mesh.centroid)

print(f"Mesh loaded — {len(mesh.vertices)} vertices, extents: {mesh.extents.round(3)}")

Mesh loaded — 3858722 vertices, extents: [0.456 0.425 0.438]


In [ ]:
from scipy.spatial import ConvexHull

def render_fast(mesh, azimuth_deg, elevation_deg=30, img_size=128, padding=12):
    rot_az = trimesh.transformations.rotation_matrix(np.radians(azimuth_deg), [0, 0, 1])
    rot_el = trimesh.transformations.rotation_matrix(np.radians(elevation_deg), [1, 0, 0])
    verts  = trimesh.transform_points(mesh.vertices,
                 trimesh.transformations.concatenate_matrices(rot_el, rot_az))
    xy     = verts[:, :2]
    scale  = (img_size - 2 * padding) / (xy.max(axis=0) - xy.min(axis=0)).max()
    xy_px  = ((xy - xy.min(axis=0)) * scale + padding).astype(np.float32)
    # invert Y so the image matches a top-down camera view
    xy_px[:, 1] = img_size - 1 - xy_px[:, 1]
    hull   = ConvexHull(xy_px)
    pts    = xy_px[hull.vertices].astype(np.int32)
    img    = np.zeros((img_size, img_size), dtype=np.uint8)
    cv2.fillPoly(img, [pts], 255)
    return img

def render_precise(mesh, azimuth_deg, img_size=256, padding=12):
    rot = trimesh.transformations.rotation_matrix(np.radians(azimuth_deg), [0, 0, 1])
    verts = trimesh.transform_points(mesh.vertices, rot)
    xy    = verts[:, :2]
    scale = (img_size - 2 * padding) / (xy.max(axis=0) - xy.min(axis=0)).max()
    xy_px = ((xy - xy.min(axis=0)) * scale + padding).astype(np.float32)
    # invert Y so the image matches a top-down camera view
    xy_px[:, 1] = img_size - 1 - xy_px[:, 1]
    img   = np.zeros((img_size, img_size), dtype=np.uint8)
    for face in mesh.faces:
        cv2.fillPoly(img, [xy_px[face].astype(np.int32)], 255)
    return img

# Preview
fig, axes = plt.subplots(2, 4, figsize=(14, 7))
fig.suptitle("Top: step 1 fast (convex hull, 128px) | Bottom: step 2 precise (top-down, 256px)")
for c, az in enumerate([0, 45, 90, 180]):
    axes[0, c].imshow(render_fast(mesh, az),    cmap="gray")
    axes[0, c].set_title(f"fast az={az}°");     axes[0, c].axis("off")
    axes[1, c].imshow(render_precise(mesh, az), cmap="gray")
    axes[1, c].set_title(f"precise az={az}°");  axes[1, c].axis("off")
plt.tight_layout()
plt.show()

In [4]:
# Step 1: 18 templates (one elevation @ 20° step) — fast convex hull
os.makedirs("../data/templates_wide", exist_ok=True)
meta1 = []
for az in range(0, 360, 20):
    sil  = render_fast(mesh, az, elevation_deg=30)
    path = f"../data/templates_wide/el30_az{az:03d}.png"
    cv2.imwrite(path, sil)
    meta1.append({"elevation": 30, "azimuth": az, "path": path})
with open("../data/templates_wide/metadata.json", "w") as f:
    json.dump(meta1, f, indent=2)
print(f"Step 1: saved {len(meta1)} templates")

# Step 2: 72 templates (top-down, 5° step) — precise silhouette
os.makedirs("../data/templates_topdown", exist_ok=True)
meta2 = []
for az in range(0, 360, 5):
    sil  = render_precise(mesh, az)
    path = f"../data/templates_topdown/az{az:03d}.png"
    cv2.imwrite(path, sil)
    meta2.append({"azimuth": az, "path": path})
with open("../data/templates_topdown/metadata.json", "w") as f:
    json.dump(meta2, f, indent=2)
print(f"Step 2: saved {len(meta2)} templates")

Step 1: saved 18 templates
Step 2: saved 72 templates
